In [1]:
import time
import numpy as np
import pandas as pd

from Pipeline.Model.DataSplit import DataSplit
from Pipeline.Model.EvaluationMatrix import EvaluationMatrix
from Pipeline.Model.ABC_ELM2 import ABC_ELM2

In [2]:
# Load the dataset
filePath = '../../Dataset/UCI_Gallstone_Dataset.csv'
df = pd.read_csv(filePath)

targetCol = ['Gallstone Status']
X = df.drop(targetCol, axis=1)
y = df[targetCol]
features_size = X.shape[1]

# Define the activation function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

print(f"Features Size: {features_size}")
print(f"Total Samples: {len(df)}")

Features Size: 38
Total Samples: 319


In [3]:
test_size = 0.2
k_fold = 5
random_seed = 42

splitter = DataSplit(random_state=random_seed, test_size=test_size, k_fold=k_fold)

x_test, y_test, folds = splitter.k_fold_data_spiting(X, y)

Holdout Test Set Size: 64
Number of CV Folds: 5


In [4]:
# Define ABC_ELM2 Hyperparameters (using best values found in standard ELM grid search as a baseline)
hidden_size_opt = 60
lambda_opt = 8.75

cv_results = []

print(f"--- Starting {k_fold}-Fold Cross Validation for ABC-ELM ---")

for fold_idx in range(k_fold):
    start_time = time.time()

    fold = folds[fold_idx]
    x_train_fold, y_train_fold = fold['X_train_fold'], fold['y_train_fold']
    x_val_fold, y_val_fold     = fold['X_val_fold'], fold['y_val_fold']

    abc_elm = ABC_ELM2(
        feature_size=features_size,
        hidden_size=hidden_size_opt,
        activation_function=sigmoid,
        regularization_lambda=lambda_opt,
        algo_type='algo3',
        random_seed=random_seed + fold_idx,
        SN=10,
        limit=10,
        iter_max=50
    )

    abc_elm.fit(x_train_fold, y_train_fold)

    y_pred_val = abc_elm.predict(x_val_fold)

    # Evaluate
    eval_matrix = EvaluationMatrix(y_val_fold, y_pred_val)
    metrics = eval_matrix.get_all_metrics()
    metrics['Fold'] = fold_idx + 1
    cv_results.append(metrics)

    elapsed = time.time() - start_time
    print(f"Fold {fold_idx + 1} completed in {elapsed:.2f}s | F2-Score: {metrics['F2-Score']:.4f} | MCC: {metrics['MCC']:.4f}")

# Aggregate and display CV results
cv_df = pd.DataFrame(cv_results)
cv_summary = cv_df.drop('Fold', axis=1).mean().to_frame("CV_Mean").join(
             cv_df.drop('Fold', axis=1).std().to_frame("CV_Std"))

print("\n--- Cross-Validation Summary ---")
display(cv_summary)

--- Starting 5-Fold Cross Validation for ABC-ELM ---
Iteration 1 end : 0.4406s
Iteration 2 end : 0.4282s
Iteration 3 end : 0.4584s
Iteration 4 end : 0.3924s
Iteration 5 end : 0.3939s
Iteration 6 end : 0.4323s
Iteration 7 end : 0.3766s
Iteration 8 end : 0.3856s
Iteration 9 end : 0.3807s
Iteration 10 end : 0.3491s
Iteration 11 end : 0.3525s
Iteration 12 end : 0.3505s
Iteration 13 end : 0.3675s
Iteration 14 end : 0.3607s
Iteration 15 end : 0.3204s
Iteration 16 end : 0.3086s
Iteration 17 end : 0.2900s
Iteration 18 end : 0.2918s
Iteration 19 end : 0.2813s
Iteration 20 end : 0.3231s
Iteration 21 end : 0.2692s
Iteration 22 end : 0.3067s
Iteration 23 end : 0.3198s
Iteration 24 end : 0.2587s
Iteration 25 end : 0.2939s
Iteration 26 end : 0.2528s
Iteration 27 end : 0.2413s
Iteration 28 end : 0.2506s
Iteration 29 end : 0.2124s
Iteration 30 end : 0.2019s
Iteration 31 end : 0.1973s
Iteration 32 end : 0.1803s
Iteration 33 end : 0.1694s
Iteration 34 end : 0.2075s
Iteration 35 end : 0.1625s
Iteration 3

,CV_Mean,CV_Std
Accuracy,0.701961,0.110398
Precision,0.762302,0.146917
Recall,0.596000,0.120333
NPV,0.671012,0.098450
Specificity,0.806154,0.143951
F1-Score,0.664120,0.115042
F2-Score,0.620639,0.116128
Balanced Accuracy,0.701077,0.109406
MCC,0.417263,0.225424


In [7]:
print("--- Starting Final Training & Testing ---")
start_time = time.time()

# To evaluate on the final test set, we need the full scaled training set
# DataSplit stores the main scaler and the full scaled train/test sets internally
x_train_scaled = splitter.x_train_scaled
y_train_full = y.drop(x_test.index).reset_index(drop=True) # Ensure matching indices
x_test_scaled = splitter.x_test_scaled

# Instantiate the final model
final_abc_elm = ABC_ELM2(
    feature_size=features_size,
    hidden_size=hidden_size_opt,
    activation_function=sigmoid,
    regularization_lambda=lambda_opt,
    algo_type='algo3',
    random_seed=random_seed,
    SN=20,          # Increased swarm size for final training
    limit=15,
    iter_max=100    # Increased iterations for final convergence
)

print("Training final ABC-ELM model on full training set...")
final_abc_elm.fit(x_train_scaled, y_train_full)

print("Predicting on unseen holdout test set...")
y_pred_test = final_abc_elm.predict(x_test_scaled)

# Final Evaluation
final_eval = EvaluationMatrix(y_test, y_pred_test)
report = final_eval.get_report()

elapsed = time.time() - start_time
print(f"Final testing completed in {elapsed:.2f}s\n")

print("--- Final Holdout Test Report ---")
print(f"Confusion Matrix Counts: {report['Counts']}")
display(pd.DataFrame([report['Metrics']]).T.rename(columns={0: "Test Metrics"}))

--- Starting Final Training & Testing ---
Training final ABC-ELM model on full training set...
Iteration 1 end : 1.1479s
Iteration 2 end : 1.2950s
Iteration 3 end : 1.2879s
Iteration 4 end : 1.2762s
Iteration 5 end : 1.3149s
Iteration 6 end : 1.2673s
Iteration 7 end : 1.2854s
Iteration 8 end : 1.2322s
Iteration 9 end : 1.2459s
Iteration 10 end : 1.1863s
Iteration 11 end : 1.2200s
Iteration 12 end : 1.2376s
Iteration 13 end : 1.2348s
Iteration 14 end : 1.1619s
Iteration 15 end : 1.2226s
Iteration 16 end : 1.2105s
Iteration 17 end : 1.2048s
Iteration 18 end : 1.1503s
Iteration 19 end : 1.1843s
Iteration 20 end : 1.1145s
Iteration 21 end : 1.0731s
Iteration 22 end : 1.0404s
Iteration 23 end : 1.0879s
Iteration 24 end : 1.0678s
Iteration 25 end : 1.0395s
Iteration 26 end : 1.0561s
Iteration 27 end : 1.0420s
Iteration 28 end : 1.0158s
Iteration 29 end : 1.0541s
Iteration 30 end : 1.0119s
Iteration 31 end : 0.9657s
Iteration 32 end : 0.9780s
Iteration 33 end : 0.9032s
Iteration 34 end : 0.91

,Test Metrics
Accuracy,0.4219
Precision,0.4468
Recall,0.6562
NPV,0.3529
Specificity,0.1875
F1-Score,0.5316
F2-Score,0.6000
Balanced Accuracy,0.4219
MCC,-0.1769
